# Práctica 2 — Transfer Learning con MobileNetV2
**Materia:** Deep Learning  
**Maestría en Ciencias en Inteligencia Artificial — UAQ**

## Objetivo
Aplicar transfer learning usando MobileNetV2 preentrenado en ImageNet para clasificar el dataset CIFAR-10, comparando el entrenamiento desde cero contra el ajuste fino (fine-tuning).

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import numpy as np
import matplotlib.pyplot as plt

CLASES = ['avión','auto','pájaro','gato','ciervo','perro','rana','caballo','barco','camión']
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
X_train = tf.image.resize(X_train, [96, 96]).numpy() / 255.0
X_test  = tf.image.resize(X_test,  [96, 96]).numpy() / 255.0
y_train_cat = to_categorical(y_train.flatten(), 10)
y_test_cat  = to_categorical(y_test.flatten(),  10)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

## 2. Construcción del modelo con base preentrenada

In [ ]:
base = MobileNetV2(input_shape=(96,96,3), include_top=False, weights='imagenet')
base.trainable = False

inputs = tf.keras.Input(shape=(96,96,3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs, outputs, name='MobileNetV2_CIFAR10')
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 3. Fase 1 — Entrenamiento de la cabeza clasificadora

In [ ]:
h1 = model.fit(X_train[:10000], y_train_cat[:10000], epochs=5,
               batch_size=64, validation_split=0.1, verbose=1)
print(f"Acc validación fase 1: {max(h1.history['val_accuracy'])*100:.2f}%")

## 4. Fase 2 — Fine-tuning de las últimas capas

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy', metrics=['accuracy'])

h2 = model.fit(X_train[:10000], y_train_cat[:10000], epochs=5,
               batch_size=32, validation_split=0.1, verbose=1)
print(f"Acc validación fase 2: {max(h2.history['val_accuracy'])*100:.2f}%")

In [ ]:
loss, acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Precisión final en prueba: {acc*100:.2f}%")

## Conclusiones
- Transfer learning permite aprovechar representaciones aprendidas en grandes datasets (ImageNet) para tareas específicas.
- La fase de fine-tuning desbloquea las capas finales para ajustarse al dominio objetivo.
- Este paradigma reduce drásticamente el tiempo de entrenamiento y los requisitos de datos etiquetados.